In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import gc
import json
import torch
import subprocess
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,    
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup
)
from torchcrf import CRF

# --- High Speed P40 Settings ---
# Use "xlm-roberta-base" or point to your existing local BERT directory
MODEL_PATH = "./xlm-robert-base"
OUTPUT_DIR = "./bert_crf_model"
CHECKPOINT_DIR = "./bert_training_checkpoint"
DATA_CACHE_DIR = "./bert_dataset_cache"

TRAIN_FILE = "bilstm_data/bilstm_train.jsonl"
VAL_FILE = "bilstm_data/bilstm_validation.jsonl"
TEST_FILE = "bilstm_data/bilstm_test.jsonl"

EPOCHS = 4
LEARNING_RATE = 4e-5
WEIGHT_DECAY = 0.01

# --- Fast Batching Restored ---
BATCH_SIZE = 64             # Huge batch size for rapid P40 processing
ACCUMULATION_STEPS = 1      
MAX_LEN = 128               # Safe sequence length for NER
SAVE_STEPS = 500            

class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path):
        super().__init__()
        self.config = config
        # 1. Base Transformer (BERT, RoBERTa, etc.)
        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        
        # 2. Linear Classifier 
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        
        # 3. CRF Layer
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        
        emissions = self.classifier(sequence_output)
        
        # Convert attention_mask to uint8 to prevent pytorch-crf crashes on PyTorch 2.x
        mask = attention_mask.type(torch.uint8)
        
        if labels is not None:
            # CRF crashes if it sees the HF default padding tag (-100). 
            # We map -100 to 0. The attention_mask ensures the CRF ignores these tokens anyway.
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            
            # CRF returns the log-likelihood. We multiply by -1 to get the loss.
            loss = -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
            return loss
        else:
            # Viterbi decoding for inference
            return self.crf.decode(emissions, mask=mask)
            
    def save_pretrained(self, save_directory):
        """Custom save method to stay compatible with Hugging Face logic."""
        os.makedirs(save_directory, exist_ok=True)
        self.config.save_pretrained(save_directory)
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
        # Added utilization.gpu to the query
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        
        best_id = -1
        max_free_mb = 0
        
        # Fallback variables in case ALL GPUs are >50% utilized
        fallback_id = 0
        fallback_max_mb = 0
        
        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gpu_id = int(parts[0])
            free_memory = int(parts[1])
            gpu_util = int(parts[2])  # Extracted GPU utilization
            
            print(f"  GPU {gpu_id}: {free_memory} MB free | {gpu_util}% util")
            
            # Track the highest memory GPU globally as a fallback
            if free_memory > fallback_max_mb:
                fallback_max_mb = free_memory
                fallback_id = gpu_id
                
            # Primary logic: Only consider GPUs with < 50% utilization
            if gpu_util < 30:
                if free_memory > max_free_mb:
                    max_free_mb = free_memory
                    best_id = gpu_id
                    
        # Final Selection
        if best_id != -1:
            print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM and low utilization.\n")
            return torch.device(f"cuda:{best_id}")
        else:
            print(f"⚠️ All GPUs are highly utilized (>= 30%). Falling back to GPU {fallback_id} with {fallback_max_mb} MB free.\n")
            return torch.device(f"cuda:{fallback_id}")
            
    except Exception as e:
        print(f"⚠️ Failed to query nvidia-smi: {e}. Falling back to cuda:0.")
        return torch.device("cuda:0")

def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    
    field_to_tag = {
        "flat": "UNIT",
        "floor": "FLOOR",
        "block": "BLOCK",             # <-- ADD THIS
        "phase": "PHASE",             # <-- ADD THIS
        "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT",
        "district": "LOCATION",
        "region": "REGION",
        "village_name": "VILLAGE_NAME", 
        "building_number": "BUILDING_NUMBER"
    }
    
    # Flatten line1 and line2 into a single dictionary
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict

    # 1. Gather all terms to process
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
            
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
                
    # 2. CRITICAL FIX: Sort by length of term descending. 
    # This ensures "MUI WO KAU TSUEN" is processed before "MUI WO"
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    
    # 3. Apply tags while avoiding overwriting
    for field, tag, term in items_to_process:
        start_idx = 0
        while True:
            idx = input_text.find(term, start_idx)
            if idx == -1:
                break # Term not found or no more occurrences
            
            # Check if this specific occurrence is already tagged by a longer entity
            is_already_tagged = any(char_labels[i] != "O" for i in range(idx, idx + len(term)))
            
            if not is_already_tagged:
                # Mark B- and I- tags on this specific untagged span
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break # Successfully tagged, move to the next term in items_to_process
            else:
                # This occurrence was already tagged, keep searching forward!
                start_idx = idx + 1
                
    return char_labels


def parse_and_tokenize(tokenizer):
    print("📂 Parsing datasets and reconstructing NER labels...")
    
    unique_labels = {"O"}
    # --- UPDATE THIS LIST ---
    tag_list = ["UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME", 
                "SUB_DISTRICT", "LOCATION", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
                "BLOCK", "PHASE"]     # <-- ADD YOUR NEW LABELS HERE
                
    for tag in tag_list:
        unique_labels.add(f"B-{tag}")
        unique_labels.add(f"I-{tag}")
        
    label_list = sorted(list(unique_labels))
    label_to_id = {l: i for i, l in enumerate(label_list)}

    datasets_dict = {}
    for file_path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: {file_path} not found. Skipping...")
            continue
            
        texts, labels_list = [], []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, desc=f"Reading {file_path}"):
                line = line.strip()
                if not line: continue
                data = json.loads(line)
                
                input_text = data.get("input", "")
                output_dict = data.get("output", {})
                
                char_labels = reconstruct_char_labels(input_text, output_dict)
                texts.append(input_text)
                labels_list.append(char_labels)
                
        datasets_dict[file_path] = Dataset.from_dict({"text": texts, "char_labels": labels_list})

    os.makedirs(DATA_CACHE_DIR, exist_ok=True)
    with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "w") as f:
        json.dump({"label_list": label_list, "label_to_id": label_to_id}, f)

    def align_labels(examples):
        tokenized = tokenizer(examples["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
        labels = []
        
        for i, offsets in enumerate(tokenized["offset_mapping"]):
            char_labels = examples["char_labels"][i]
            
            token_labels = []
            for start_offset, end_offset in offsets:
                if start_offset == end_offset: 
                    token_labels.append(-100)
                else:
                    tag = "O"
                    for char_idx in range(start_offset, end_offset):
                        if char_idx < len(char_labels) and char_labels[char_idx] != "O":
                            tag = char_labels[char_idx]
                            break
                    token_labels.append(label_to_id[tag])
                    
            labels.append(token_labels)
            
        tokenized["labels"] = labels
        tokenized.pop("offset_mapping") 
        return tokenized

    print("⏳ Tokenizing datasets...")
    train_ds = datasets_dict[TRAIN_FILE].map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    val_ds = datasets_dict.get(VAL_FILE)
    if val_ds:
        val_ds = val_ds.map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    
    print(f"💾 Caching tokenized datasets to {DATA_CACHE_DIR}...")
    train_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "train"))
    if val_ds:
        val_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "val"))
    
    return train_ds, val_ds, label_list, label_to_id

def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss):
    print(f"\n💾 Saving checkpoint at Epoch {epoch+1}, Step {step}...")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    model.save_pretrained(CHECKPOINT_DIR)
    tokenizer.save_pretrained(CHECKPOINT_DIR)
    
    state = {
        "epoch": epoch,
        "step": step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val_loss
    }
    torch.save(state, os.path.join(CHECKPOINT_DIR, "training_state.pt"))
    print("✅ Checkpoint saved safely.")

def main():
    device = get_emptiest_gpu_safely()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    
    # --- 1. Smart Dataset Loading ---
    if os.path.exists(os.path.join(DATA_CACHE_DIR, "train")):
        print(f"⚡ Found cached datasets in {DATA_CACHE_DIR}. Loading instantly...")
        train_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "train"))
        val_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "val"))
        
        with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "r") as f:
            label_data = json.load(f)
            label_list = label_data["label_list"]
            label_to_id = {k: int(v) for k, v in label_data["label_to_id"].items()}
    else:
        train_ds, val_ds, label_list, label_to_id = parse_and_tokenize(tokenizer)

    print("⏳ Initializing NER Architecture...")
    # --- 2. Smart Model Resuming ---
    resume_from_checkpoint = os.path.exists(os.path.join(CHECKPOINT_DIR, "training_state.pt"))
    
    # Load config dynamically
    config = AutoConfig.from_pretrained(
        MODEL_PATH, 
        num_labels=len(label_list), 
        id2label={i: l for l, i in label_to_id.items()},
        label2id=label_to_id
    )
    
    # Initialize custom BERT+CRF Model
    model = BertCRFForTokenClassification(config, MODEL_PATH)

    # Load custom weights if resuming
    if resume_from_checkpoint:
        weights_path = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")
        if os.path.exists(weights_path):
            model.load_state_dict(torch.load(weights_path, map_location=device))

    print("🔥 Running Full Fine-Tuning (No Frozen Layers)...")
    # print("🥶 Freezing bottom layers...")
    # Generic freezing logic compatible with both RoBERTa and BERT
    # base_model = getattr(model, model.base_model_prefix, model)
    # if hasattr(base_model, "embeddings"):
    #     for param in base_model.embeddings.parameters():
    #         param.requires_grad = False
            
    # if hasattr(base_model, "encoder"):
    #     encoder_layers = base_model.encoder.layer
    #     num_layers_to_freeze = max(0, len(encoder_layers) - 4) 
    #     for layer in encoder_layers[:num_layers_to_freeze]:
    #         for param in layer.parameters():
    #             param.requires_grad = False

    # if hasattr(model, "enable_input_require_grads"):
    #     model.enable_input_require_grads()

    model.to(device)

    # --- 3. Data Loaders ---
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) 
    
    train_loader = DataLoader(
        train_ds, 
        shuffle=True, 
        batch_size=BATCH_SIZE, 
        collate_fn=data_collator,
        num_workers=4,        
        pin_memory=True        
    )
    val_loader = DataLoader(
        val_ds, 
        batch_size=BATCH_SIZE * 2, 
        collate_fn=data_collator,
        num_workers=4,
        pin_memory=True
    ) if val_ds else None
    
    torch.cuda.empty_cache()
    gc.collect()
    
    # Optimizer and Scheduler
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
    
    start_epoch = 0
    start_step = 0
    best_val_loss = float('inf')
    
    # --- 4. Restore Training State ---
    if resume_from_checkpoint:
        print(f"🔄 Resuming from saved checkpoint at {CHECKPOINT_DIR}...")
        state = torch.load(os.path.join(CHECKPOINT_DIR, "training_state.pt"), map_location=device)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch = state["epoch"]
        start_step = state["step"]
        best_val_loss = state.get("best_val_loss", float('inf'))
        print(f"⏩ Fast-forwarding to Epoch {start_epoch+1}, Step {start_step}...")
        
    print("\n🚀 Training Initiated...")
    
    # --- 5. Graceful Interruption Wrapper ---
    try:
        epoch_pbar = tqdm(range(start_epoch, EPOCHS), desc="Epochs", initial=start_epoch, total=EPOCHS)
        for epoch in epoch_pbar:
            model.train()
            total_train_loss = 0
            optimizer.zero_grad() 
            
            batch_pbar = tqdm(train_loader, desc=f"Train (Ep {epoch+1})", leave=False)
            for step, batch in enumerate(batch_pbar):
                if epoch == start_epoch and step < start_step:
                    # tqdm already updates automatically on every loop iteration, 
                    # so we just need to continue to skip processing the batch!
                    # batch_pbar.update(1)
                    continue
                
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                
                loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = loss / ACCUMULATION_STEPS
                loss.backward()
                
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                
                display_loss = loss.item() * ACCUMULATION_STEPS
                total_train_loss += display_loss
                batch_pbar.set_postfix({"loss": f"{display_loss:.4f}"})
                
                if (step + 1) % SAVE_STEPS == 0:
                    save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step + 1, best_val_loss)
                    
            avg_train_loss = total_train_loss / (len(train_loader) - (start_step if epoch == start_epoch else 0))
            start_step = 0 
            
            # --- VALIDATION ---
            if val_loader:
                model.eval()
                total_val_loss = 0
                with torch.no_grad():
                    val_pbar = tqdm(val_loader, desc=f"Val (Ep {epoch+1})", leave=False)
                    for batch in val_pbar:
                        input_ids = batch["input_ids"].to(device)
                        attention_mask = batch["attention_mask"].to(device)
                        labels = batch["labels"].to(device)
                        
                        val_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                        total_val_loss += val_loss.item()
                        
                avg_val_loss = total_val_loss / len(val_loader)
                epoch_pbar.set_postfix({"Train Loss": f"{avg_train_loss:.4f}", "Val Loss": f"{avg_val_loss:.4f}"})
                
                save_checkpoint(model, tokenizer, optimizer, scheduler, epoch + 1, 0, best_val_loss)
                
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    print(f"\n🌟 New best validation loss ({best_val_loss:.4f})! Exporting to {OUTPUT_DIR}...")
                    os.makedirs(OUTPUT_DIR, exist_ok=True)
                    # For torch.compile models, grab the inner model weights safely
                    uncompiled_model = getattr(model, "_orig_mod", model)
                    uncompiled_model.save_pretrained(OUTPUT_DIR)
                    tokenizer.save_pretrained(OUTPUT_DIR)

        print(f"✅ Training complete. Best model saved to {OUTPUT_DIR}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Training Interrupted by User (Ctrl+C)!")
        save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss)
        print("👋 Safe exit complete. Run the script again to resume from this exact spot.")
        sys.exit(0)

if __name__ == "__main__":
    main()


🔍 Scanning available GPUs safely via nvidia-smi...
  GPU 0: 9227 MB free | 100% util
  GPU 1: 20420 MB free | 0% util
  GPU 2: 21610 MB free | 0% util
  GPU 3: 6400 MB free | 0% util
--> Selected GPU 2 with 21610 MB free VRAM and low utilization.

📂 Parsing datasets and reconstructing NER labels...


Reading bilstm_data/bilstm_train.jsonl: 0it [00:00, ?it/s]

Reading bilstm_data/bilstm_validation.jsonl: 0it [00:00, ?it/s]

Reading bilstm_data/bilstm_test.jsonl: 0it [00:00, ?it/s]

⏳ Tokenizing datasets...


...


💾 Saving checkpoint at Epoch 4, Step 0...
✅ Checkpoint saved safely.

🌟 New best validation loss (0.5640)! Exporting to ./bert_crf_model...


Train (Ep 4):   0%|          | 0/2602 [00:00<?, ?it/s]


💾 Saving checkpoint at Epoch 4, Step 500...
✅ Checkpoint saved safely.

💾 Saving checkpoint at Epoch 4, Step 1000...
✅ Checkpoint saved safely.

💾 Saving checkpoint at Epoch 4, Step 1500...
✅ Checkpoint saved safely.

💾 Saving checkpoint at Epoch 4, Step 2000...
✅ Checkpoint saved safely.

💾 Saving checkpoint at Epoch 4, Step 2500...
✅ Checkpoint saved safely.


Val (Ep 4):   0%|          | 0/163 [00:00<?, ?it/s]


💾 Saving checkpoint at Epoch 5, Step 0...
✅ Checkpoint saved safely.

🌟 New best validation loss (0.4754)! Exporting to ./bert_crf_model...
✅ Training complete. Best model saved to ./bert_crf_model


In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./bert_crf_model" 
LOG_FILE = "parsing_results_bert.log"
TEST_FILE = "bilstm_data/bilstm_test.jsonl" 
MAX_LEN = 128
# ==========================================

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path):
        super().__init__()
        self.config = config
        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)
        
        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
        else:
            return self.crf.decode(emissions, mask=mask)

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        print("CUDA not available. Falling back to CPU (-1).")
        return -1
        
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip(): continue
            gpu_id, free_memory = line.split(", ")
            gpu_id, free_memory = int(gpu_id), int(free_memory)
            
            if free_memory > max_free_mb:
                max_free_mb = free_memory
                best_id = gpu_id
                
        return best_id
    except Exception:
        return 0

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    for entity in parsed_entities:
        tag = entity['entity_group']
        if tag == 'O': 
            continue
            
        word = entity['word'].strip()
        if word:
            components[tag].append(word)

    formatted_output = {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any('\u4e00' <= char <= '\u9fff' for char in joined_string):
            formatted_output[tag] = "".join(words) 
        else:
            joined_en = " ".join(words).strip()
            # Updated fix for slashes, hyphens, and periods
            joined_en = re.sub(r'\s*([/\.-])\s*', r'\1', joined_en)
            formatted_output[tag] = joined_en
            
    return formatted_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""),
            "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""),       # <-- ADD THIS
            "phase": extracted_data.get("PHASE", ""),       # <-- ADD THIS
            "building_name": extracted_data.get("BUILDING_NAME", "")
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""),
            "village_name": extracted_data.get("VILLAGE_NAME", ""),       
            "building_number": extracted_data.get("BUILDING_NUMBER", ""), 
            "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""), 
            "district": extracted_data.get("LOCATION", ""),
            "region": extracted_data.get("REGION", "")
        }
    }

def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    total_time = 0
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")
    
    print(f"DEBUG: Loading Config, Tokenizer, and BERT+CRF Model from {MODEL_DIR}...")
    
    if not os.path.exists(MODEL_DIR):
        print(f"❌ Error: {MODEL_DIR} not found.")
        return
        
    config = AutoConfig.from_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    
    # Initialize architecture
    model = BertCRFForTokenClassification(config, MODEL_DIR)
    
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    else:
        print(f"⚠️ Warning: Weights not found at {weights_path}.")
    
    model.to(device)
    model.eval()

    if not os.path.exists(TEST_FILE):
        print(f"❌ Error: Dataset file '{TEST_FILE}' not found.")
        return

    # Tracking Metrics
    all_fields = [
        "flat", "floor", "building_name", 
        "block", "phase",                             # <-- ADD THEM HERE
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    total_samples = 0
    exact_matches = 0
    field_correct = {field: 0 for field in all_fields}

    print("🚀 Running evaluation and accuracy check...")
    with open(TEST_FILE, "r", encoding="utf-8") as file, \
         open(LOG_FILE, "w", encoding="utf-8") as log:
        
        lines = [line for line in file if line.strip() and not line.startswith('#')]
        
        for line in tqdm(lines, desc="Evaluating"):
            data = json.loads(line)
            address = data["input"].strip()
            ground_truth_raw = data.get("output", {})
            ground_truth_flat = flatten_json(ground_truth_raw)
            
            start_time = time.perf_counter()
                
            # --- TOKENIZATION & INFERENCE ---
            encoded = tokenizer(
                address, 
                truncation=True, 
                max_length=MAX_LEN, 
                return_offsets_mapping=True, 
                return_tensors="pt"
            )
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            offsets = encoded["offset_mapping"][0].tolist()

            with torch.no_grad():
                prediction_ids = model(input_ids=input_ids, attention_mask=attention_mask)[0]
            
            # --- MAP TO CHARACTERS (Bridging BERT subwords to Regex words) ---
            char_tags = ["O"] * len(address)
            for idx, tag_id in enumerate(prediction_ids):
                start, end = offsets[idx]
                if start == end: 
                    continue # Skip special tokens like [CLS], [SEP]
                
                # Fetch tag from config.id2label. (Ensure keys are integers)
                tag = config.id2label[tag_id] if isinstance(tag_id, int) else config.id2label[str(tag_id)]
                
                # Assign tag to original characters
                for c in range(start, end):
                    if char_tags[c] == "O": # Only assign if not already filled
                        char_tags[c] = tag

            # --- EXTRACT VIA REGEX (Identical to BiLSTM logic) ---
            parsed_entities = []
            for match in re.finditer(r'[a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s]', address):
                token_str = match.group()
                start_idx = match.start()
                
                tag = char_tags[start_idx]
                entity_group = tag.replace("B-", "").replace("I-", "")
                if tag == "O": 
                    entity_group = "O"
                    
                parsed_entities.append({"entity_group": entity_group, "word": token_str})
            
            extracted_data = extract_3d_components(parsed_entities)
            final_json = assemble_compact_json(extracted_data)
            predicted_flat = flatten_json(final_json)
            
            total_time += (time.perf_counter() - start_time)

            # --- ACCURACY COMPARISON ---
            total_samples += 1
            is_perfect_match = True
            
            for field in all_fields:
                pred_val = predicted_flat.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                
                if pred_val == gt_val:
                    field_correct[field] += 1
                else:
                    is_perfect_match = False
                    
            if is_perfect_match:
                exact_matches += 1

            # --- LOGGING ---
            log.write(f"Original: {address}\n")
            if is_perfect_match:
                log.write("✅ EXACT MATCH\n")
            else:
                log.write("❌ MISMATCH FOUND\n")
                
            for field in all_fields:
                pred_val = predicted_flat.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if pred_val == gt_val else "❌"
                    log.write(f"  {status} {field.upper()}:\n")
                    log.write(f"      PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f"      TRUE: {gt_val if gt_val else '[None]'}\n")
            log.write("-" * 50 + "\n")

    # --- PRINT ACCURACY REPORT ---
    print("\n" + "="*40)
    print("📊 EVALUATION RESULTS")
    print("="*40)
    print(f"Total Samples Tested : {total_samples}")
    if total_samples > 0:
        exact_match_acc = (exact_matches / total_samples) * 100
        print(f"Exact Match Accuracy : {exact_match_acc:.2f}% ({exact_matches}/{total_samples} perfect addresses)")
        print("\n--- Field-Level Accuracy ---")
        for field in all_fields:
            acc = (field_correct[field] / total_samples) * 100
            print(f"{field.rjust(15)} : {acc:.2f}% ({field_correct[field]}/{total_samples})")
    
    print("="*40)
    print(f"✅ Processing complete. Detailed line-by-line results saved to {LOG_FILE}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds")

if __name__ == "__main__":
    main()

DEBUG: Loading Config, Tokenizer, and BERT+CRF Model from ./bert_crf_model...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).


Loading weights: 0it [00:00, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: ./bert_crf_model
Key                                                           | Status     | 
--------------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.output.LayerNorm.bias             | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.value.bias         | UNEXPECTED | 
bert.encoder.layer.{0...11}.intermediate.dense.weight         | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.key.bias           | UNEXPECTED | 
bert.encoder.layer.{0...11}.intermediate.dense.bias           | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.key.weight         | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.value.weight       | UNEXPECTED | 
bert.encoder.layer.{0...11}.output.dense.weight               | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.query.bias         | UNEXPECTED | 
bert.encoder.layer.{0...11}.output.dense.bias                 | UNEXPECTED |

🚀 Running evaluation and accuracy check...


Evaluating:   0%|          | 0/20814 [00:00<?, ?it/s]

/home/slhui.censtatd/.local/lib/python3.12/site-packages/torchcrf/__init__.py:305: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:611.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)



📊 EVALUATION RESULTS
Total Samples Tested : 20814
Exact Match Accuracy : 71.73% (14930/20814 perfect addresses)

--- Field-Level Accuracy ---
           flat : 92.45% (19243/20814)
          floor : 88.74% (18471/20814)
  building_name : 97.68% (20332/20814)
          block : 99.39% (20687/20814)
          phase : 99.26% (20661/20814)
    estate_name : 98.16% (20432/20814)
   village_name : 98.90% (20585/20814)
building_number : 97.51% (20295/20814)
    street_name : 98.98% (20602/20814)
   sub_district : 97.43% (20280/20814)
       district : 97.60% (20314/20814)
         region : 96.18% (20019/20814)
✅ Processing complete. Detailed line-by-line results saved to parsing_results_bert.log
⏱️ Total Inference runtime: 219.1885 seconds


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,  
    DataCollatorForTokenClassification, get_linear_schedule_with_warmup
)
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
# 1. Point this to the model you JUST finished training
BASE_MODEL_PATH = "./bert_crf_model" 

# 2. Point this to a COMPLETELY NEW directory to protect the original model
GOLD_OUTPUT_DIR = "./bert_crf_gold_finetuned"
CHECKPOINT_DIR = "./bert_gold_checkpoint"

# 3. Your new curated dataset
GOLD_FILE = "bilstm_data/gold_hard_cases.jsonl"

# --- FINE TUNING TOGGLES ---
FREEZE_BERT = True          # True = Method A (Head-Only), False = Method B (Micro-LR)
EPOCHS = 5
LEARNING_RATE = 5e-4 if FREEZE_BERT else 5e-6  # Use higher LR if only training CRF head
BATCH_SIZE = 16             # Smaller batch size for small datasets
ACCUMULATION_STEPS = 1
MAX_LEN = 128
WEIGHT_DECAY = 0.01

# [KEEP YOUR BertCRFForTokenClassification CLASS EXACTLY AS IS HERE]
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path):
        super().__init__()
        self.config = config
        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)
        
        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            loss = -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)
            
    def save_pretrained(self, save_directory):
        os.makedirs(save_directory, exist_ok=True)
        self.config.save_pretrained(save_directory)
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

# [KEEP YOUR reconstruct_char_labels EXACTLY AS IS HERE]
def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    
    field_to_tag = {
        "flat": "UNIT",
        "floor": "FLOOR",
        "block": "BLOCK",             # <-- ADD THIS
        "phase": "PHASE",             # <-- ADD THIS
        "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT",
        "district": "LOCATION",
        "region": "REGION",
        "village_name": "VILLAGE_NAME", 
        "building_number": "BUILDING_NUMBER"
    }
    
    # Flatten line1 and line2 into a single dictionary
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict

    # 1. Gather all terms to process
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
            
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
                
    # 2. CRITICAL FIX: Sort by length of term descending. 
    # This ensures "MUI WO KAU TSUEN" is processed before "MUI WO"
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    
    # 3. Apply tags while avoiding overwriting
    for field, tag, term in items_to_process:
        start_idx = 0
        while True:
            idx = input_text.find(term, start_idx)
            if idx == -1:
                break # Term not found or no more occurrences
            
            # Check if this specific occurrence is already tagged by a longer entity
            is_already_tagged = any(char_labels[i] != "O" for i in range(idx, idx + len(term)))
            
            if not is_already_tagged:
                # Mark B- and I- tags on this specific untagged span
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break # Successfully tagged, move to the next term in items_to_process
            else:
                # This occurrence was already tagged, keep searching forward!
                start_idx = idx + 1
                
    return char_labels

def load_and_split_gold_data(tokenizer):
    print(f"📂 Parsing Gold dataset from {GOLD_FILE}...")
    texts, labels_list = [], []
    
    # We must load the EXACT label map from your previous training
    with open("./bert_dataset_cache/label_map.json", "r") as f:
        label_data = json.load(f)
        label_list = label_data["label_list"]
        label_to_id = {k: int(v) for k, v in label_data["label_to_id"].items()}

    with open(GOLD_FILE, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading Gold File"):
            line = line.strip()
            if not line: continue
            data = json.loads(line)
            input_text = data.get("input", "")
            output_dict = data.get("output", {})
            char_labels = reconstruct_char_labels(input_text, output_dict)
            texts.append(input_text)
            labels_list.append(char_labels)

    # 1. Create a full Hugging Face dataset
    full_ds = Dataset.from_dict({"text": texts, "char_labels": labels_list})
    
    # 2. Split it 85/15 automatically
    split_ds = full_ds.train_test_split(test_size=0.15, seed=42)
    train_ds = split_ds["train"]
    val_ds = split_ds["test"]

    # 3. Alignment function mapping
    def align_labels(examples):
        tokenized = tokenizer(examples["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
        labels = []
        for i, offsets in enumerate(tokenized["offset_mapping"]):
            char_labels = examples["char_labels"][i]
            token_labels = []
            for start_offset, end_offset in offsets:
                if start_offset == end_offset: 
                    token_labels.append(-100)
                else:
                    tag = "O"
                    for char_idx in range(start_offset, end_offset):
                        if char_idx < len(char_labels) and char_labels[char_idx] != "O":
                            tag = char_labels[char_idx]
                            break
                    token_labels.append(label_to_id.get(tag, label_to_id["O"]))
            labels.append(token_labels)
        tokenized["labels"] = labels
        tokenized.pop("offset_mapping") 
        return tokenized

    print("⏳ Tokenizing Gold sets...")
    train_ds = train_ds.map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    val_ds = val_ds.map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    
    return train_ds, val_ds, label_list, label_to_id

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
    
    # Load and split the 500-1000 items
    train_ds, val_ds, label_list, label_to_id = load_and_split_gold_data(tokenizer)

    print("⏳ Loading Pre-Trained Model from Base Path...")
    config = AutoConfig.from_pretrained(BASE_MODEL_PATH)
    model = BertCRFForTokenClassification(config, BASE_MODEL_PATH)
    
    # Load the trained custom weights
    model.load_state_dict(torch.load(os.path.join(BASE_MODEL_PATH, "pytorch_model.bin"), map_location=device))

    # --- IMPLEMENTING THE TUNING METHOD ---
    if FREEZE_BERT:
        print("🥶 Method A: Freezing BERT layers. Training ONLY the Linear & CRF Heads...")
        for param in model.bert.parameters():
            param.requires_grad = False
    else:
        print("🔥 Method B: Full network tuning with Micro-Learning Rate...")

    model.to(device)

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) 
    train_loader = DataLoader(train_ds, shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, collate_fn=data_collator)

    # Pass only requires_grad=True parameters to optimizer
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    best_val_loss = float('inf')
    
    print("\n🚀 Gold Fine-Tuning Initiated...")
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        
        batch_pbar = tqdm(train_loader, desc=f"Train (Ep {epoch+1})", leave=False)
        for batch in batch_pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            optimizer.zero_grad()
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
            batch_pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                val_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += val_loss.item()
                
        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"🌟 New best gold validation! Exporting to {GOLD_OUTPUT_DIR}...")
            # Uncompiled wrapper safe-save
            uncompiled_model = getattr(model, "_orig_mod", model)
            uncompiled_model.save_pretrained(GOLD_OUTPUT_DIR)
            tokenizer.save_pretrained(GOLD_OUTPUT_DIR)

    print(f"✅ Gold Tuning complete. Test your new model at {GOLD_OUTPUT_DIR}")

if __name__ == "__main__":
    main()